# 03 - Bias mitigation: pre, in and post-processing

Notebook 03 of the `callmefair` series. We take a model that is biased against a group, apply mitigation techniques one at a time, and then let `BMGridSearch` try many combinations.

**What you'll learn**
- the three stages of bias mitigation and the `BMType` members that implement them;
- how to apply a pre-processing technique (Reweighing) and a post-processing technique (Equalized Odds) *by hand* with `BMManager`;
- how `BMGridSearch` automates this for several models and technique combinations;
- how to read the results (the table feeds notebook 04).

**Prerequisites:** [00_data_and_preprocessing.ipynb](00_data_and_preprocessing.ipynb) (data, groups, label) and [01_fairness_fundamentals.ipynb](01_fairness_fundamentals.ipynb) (metrics and the fairness score 𝓕; here 0 = fair, higher = more biased).

**Runtime**: about 3-5 minutes on a laptop.

## 1. The three stages

Mitigation can intervene at three points of the ML pipeline. `BMType` lists all techniques.

| Stage | When | `BMType` | What it does |
|---|---|---|---|
| **pre** | before training: change the training data | `preReweighing` | gives each (group, label) cell an instance weight so group and label look independent |
| | | `preDisparate` | Disparate Impact Remover: edits feature values to remove group information |
| | | `preLFR` | Learning Fair Representations: learns a fair latent representation |
| **in** | during training: fairness-aware learner | `inAdversarial` | Adversarial Debiasing: a second network tries to guess the group from the predictions |
| | | `inMeta` | MetaFair classifier with a fairness constraint |
| **post** | after training: change the predictions | `posCalibrated` | Calibrated Equalized Odds |
| | | `posEqqOds` | Equalized Odds |
| | | `posROC` | Reject Option Classification: flips decisions near the threshold |

Pre and post techniques work with *any* model. In-processing techniques bring their own model.

In [1]:
from fhg_examples import setup; setup()

## 2. Data and baseline

We study **race** on the diabetes data (see notebook 00): group `race=1` (Caucasian) is the privileged reference, `race=0` the unprivileged group; the favorable label `readmitted=1` is readmission within 30 days.

For speed we use a **stratified random sample of about 20,000 rows** (stratified by label x race, fixed seed). The full 99,492 rows work exactly the same, only slower.

In [2]:
import pandas as pd
from fhg_examples.datasets import load, SPECS

spec = SPECS['diabetes']
priv, unpriv = spec.attribute('race').groups
df = load('diabetes')
df = (df.groupby([spec.label, 'race'], group_keys=False)
        .sample(frac=20_000 / len(df), random_state=0).reset_index(drop=True))
pd.crosstab(df['race'], df[spec.label], margins=True)

readmitted,0,1,All
race,,,
0,4184,518,4702
1,13570,1727,15297
All,17754,2245,19999


Every (race, label) cell keeps enough rows. Next, the **baseline**: `BMInterface` makes the stratified train/validation/test split and serves it as aif360 datasets; a plain `LogisticRegression` is trained and `BMMetrics` reports performance and fairness on the test split. The decision threshold is tuned on the validation split.

Two small helpers keep the rest of the notebook short: `fit_proba` trains the model on the current training split (optionally with sample weights), and `evaluate` turns a `BMMetrics` object into a one-row table.

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from callmefair import BMInterface, BMMetrics, BMManager, BMType, BMGridSearch

bmI = BMInterface(df, spec.label, ['race'])
CLASSES = np.array([0, 1])
COLS = ['balanced_acc', 'spd', 'disparate_impact', 'eq_opp_diff', 'fair_score']

def evaluate(bmM, name):
    """One-row report with the columns we care about."""
    row = {**bmM.get_report(), 'fair_score': bmM.get_score()['overall_score']}
    return pd.DataFrame(row, index=[name])[COLS]

def fit_proba(bmI, sample_weight=None):
    """Train logistic regression; return (val, test) probability matrices."""
    X, y = bmI.get_train_xy()
    model = LogisticRegression(max_iter=1000).fit(X, y, sample_weight=sample_weight)
    return model.predict_proba(bmI.get_val_xy()[0]), model.predict_proba(bmI.get_test_xy()[0])


In [ ]:
val_p, test_p = fit_proba(bmI)
bmM = BMMetrics(bmI, CLASSES, val_p, test_p, priv, unpriv)
baseline = evaluate(bmM, 'baseline')
baseline

Reading the row: `spd` (statistical parity difference) and `eq_opp_diff` are 0 when fair, `disparate_impact` is 1 when fair, and `fair_score` (𝓕) is 0 when fair. Details in notebook 01.

## 3. Pre-processing by hand: Reweighing

`BMManager` applies the techniques to the data held by `bmI`. **Reweighing** does not touch features or labels: it changes the training *instance weights* so that, after weighting, group and label are independent. Under-represented (group, label) cells get weight > 1.

In [4]:
bmMgr = BMManager(bmI, priv, unpriv)
bmMgr.pre_Reweighing()

train = bmI.get_train_BLD()
w = pd.DataFrame({'race': train.protected_attributes.ravel(),
                  'label': train.labels.ravel(),
                  'weight': train.instance_weights})
w.groupby(['race', 'label'])['weight'].agg(['count', 'mean'])

count   mean
race label              
0.0  0.0     3360  0.996
     1.0      408  1.037
1.0  0.0    10843  1.001
     1.0     1388  0.989

Before, every weight was 1; now they differ by group and label. The weights reach the model through `sample_weight`: we retrain with them and evaluate as before.

In [5]:
weights = bmI.get_train_BLD().instance_weights
val_p, test_p = fit_proba(bmI, sample_weight=weights)
bmM_rw = BMMetrics(bmI, CLASSES, val_p, test_p, priv, unpriv)
rw = evaluate(bmM_rw, 'preReweighing')
pd.concat([baseline, rw])

,balanced_acc,spd,disparate_impact,eq_opp_diff,fair_score
baseline,0.598,-0.094,0.804,-0.117,0.627
preReweighing,0.600,-0.033,0.929,-0.058,0.189


**Why `restore_BLD()`?** Pre-processing *replaces* the training data inside `bmI`. To start the next experiment from the original data, call `bmI.restore_BLD()`; otherwise techniques silently stack. `BMGridSearch` does this after every combination.

In [6]:
bmI.restore_BLD()
bmI.get_train_BLD().instance_weights[:3]   # back to the original weights (all 1)

array([1., 1., 1.])

## 4. Post-processing by hand: Equalized Odds

Post-processors fix the *decisions* of an already-trained model. They need to see the model's **scores** (P(favorable)) together with its thresholded labels, for both the validation set (to learn the correction) and the test set (to apply it). So we build two *prediction datasets*: copies of the validation/test datasets whose `.scores` hold the model probabilities and whose `.labels` hold the decisions at the tuned threshold (`bmM.best_class_thresh`). This is exactly what `BMGridSearch` does internally.

In [7]:
val_p, test_p = fit_proba(bmI)
bmM = BMMetrics(bmI, CLASSES, val_p, test_p, priv, unpriv)
thr = bmM.best_class_thresh

def prediction_bld(bld, proba):
    scores = proba[:, 1].reshape(-1, 1)
    bld.scores = scores
    bld.labels = np.where(scores > thr, bld.favorable_label, bld.unfavorable_label)
    return bld

val_pred = prediction_bld(bmI.get_val_BLD(), val_p)
test_pred = prediction_bld(bmI.get_test_BLD(), test_p)

In [8]:
val_post, test_post = bmMgr.pos_apply(BMType.posEqqOds, val_pred, test_pred)
bmM.set_pos_pred(test_post)          # metrics now use the post-processed decisions
pos = evaluate(bmM, 'posEqqOds')
pd.concat([baseline, rw, pos])

,balanced_acc,spd,disparate_impact,eq_opp_diff,fair_score
baseline,0.598,-0.094,0.804,-0.117,0.627
preReweighing,0.600,-0.033,0.929,-0.058,0.189
posEqqOds,0.597,-0.010,0.979,-0.047,0.119


Post-processing changes the decisions only, so it can trade accuracy for fairness. Note that Equalized Odds is *randomised* and fitted on the validation split: with very small validation groups it can degenerate (for example predicting everyone favorable or unfavorable).

## 5. Grid search: many combinations, many models

Trying techniques by hand does not scale. `BMGridSearch` takes a model and a list of combinations (`bm_list`, a list of lists of `BMType`, applied pre, then post), and returns one row per combination plus the **baseline**. It restores the data between combinations. Pass `results_dir=None` to keep results in memory only (the default also writes a CSV).

We build `bm_list` with `itertools.product`: every pre x post pair plus each technique alone. Two models are compared: `LogisticRegression` and `XGBClassifier`.

In [9]:
from itertools import product
from xgboost import XGBClassifier

pre = [BMType.preReweighing, BMType.preDisparate]
post = [BMType.posCalibrated, BMType.posEqqOds, BMType.posROC]
bm_list = [list(c) for c in product(pre, post)] + [[b] for b in pre + post]
len(bm_list), [[b.name for b in c] for c in bm_list[:3]]

(11,
 [['preReweighing', 'posCalibrated'],
  ['preReweighing', 'posEqqOds'],
  ['preReweighing', 'posROC']])

In [10]:
models = {'LogisticRegression': LogisticRegression(max_iter=1000),
          'XGBClassifier': XGBClassifier(n_estimators=200, n_jobs=-1)}
rows = []
for name, model in models.items():
    grid = BMGridSearch(bmI, model, bm_list, priv, unpriv)
    rows += grid.run_single_sensitive(results_dir=None)
res = pd.DataFrame(rows)
res.shape

(24, 14)

Rows with the lowest fairness score 𝓕 (closest to fair):

In [11]:
SHOW = ['model', 'BM', 'balanced_acc', 'spd', 'disparate_impact', 'eq_opp_diff', 'fair_score']
res.sort_values('fair_score')[SHOW].head(6).round(3).reset_index(drop=True)

,model,BM,balanced_acc,spd,disparate_impact,eq_opp_diff,fair_score
0,XGBClassifier,preReweighing posEqqOds,0.533,-0.004,0.979,-0.011,0.077
1,XGBClassifier,preDisparate posROC,0.575,0.009,1.022,-0.022,0.081
2,XGBClassifier,posEqqOds,0.564,0.013,1.033,-0.044,0.118
3,LogisticRegression,posCalibrated,0.508,0.000,1.103,-0.024,0.118
4,LogisticRegression,preReweighing posEqqOds,0.596,-0.005,0.989,-0.058,0.123
5,LogisticRegression,preDisparate posEqqOds,0.594,-0.008,0.984,-0.064,0.137


A compact view, 𝓕 by technique and model (the `baseline` row is the reference):

In [12]:
pivot = res.pivot(index='BM', columns='model', values='fair_score').round(3)
pivot.sort_values('LogisticRegression')

model,LogisticRegression,XGBClassifier
BM,,
posCalibrated,0.118,0.605
preReweighing posEqqOds,0.123,0.077
preDisparate posEqqOds,0.137,0.306
posEqqOds,0.164,0.118
preReweighing,0.189,0.303
preReweighing posROC,0.189,0.149
preDisparate posCalibrated,0.385,0.292
posROC,0.395,0.202
preDisparate posROC,0.397,0.081


The scatter below is the **fairness-accuracy trade-off**: each point is one (model, combination). Good points are left (fair) and up (accurate). Baselines are circled. Notebook 04 builds on this table.

In [13]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 4))
for name, g in res.groupby('model'):
    ax.scatter(g['fair_score'], g['balanced_acc'], label=name, alpha=.8)
    b = g[g['BM'] == 'baseline']
    ax.scatter(b['fair_score'], b['balanced_acc'], s=160, facecolors='none', edgecolors='k')
ax.set_xlabel('fair_score 𝓕 (0 = fair)'); ax.set_ylabel('balanced accuracy')
ax.legend(); plt.show()

## 6. In-processing: Adversarial Debiasing (last section)

In-processing techniques are fairness-aware learners, so `BMGridSearch` gets `model=None` and a list holding the in-processing technique. The first row (`baseline`) is the *same* network trained with `debias=False`; the `inAD` row is trained with the adversary.

> **Caveat.** `AdversarialDebiasing` uses TensorFlow 1 graph mode and **disables eager execution for the whole kernel**. That is why this section comes last: restart the kernel before running any TensorFlow 2 code. Also, `inMeta` (MetaFair) with `tau>0` currently fails inside aif360 (known issue), so we do not run it.

In [14]:
import io, contextlib
import tensorflow as tf
tf.get_logger().setLevel('ERROR')   # silence TF deprecation notices

grid = BMGridSearch(bmI, None, [[BMType.inAdversarial]], priv, unpriv)
with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    rows_ad = grid.run_single_sensitive(results_dir=None)   # hides TF training logs
ad = pd.DataFrame(rows_ad)
ad[SHOW].round(3)

,model,BM,balanced_acc,spd,disparate_impact,eq_opp_diff,fair_score
0,AdversarialDebiasing,baseline,0.567,-0.035,0.937,-0.047,0.172
1,AdversarialDebiasing,inAD,0.571,-0.001,0.997,-0.016,0.063


## Summary

- **pre** (`preReweighing`, `preDisparate`, `preLFR`) changes the training data; **in** (`inAdversarial`, `inMeta`) changes the learner; **post** (`posCalibrated`, `posEqqOds`, `posROC`) changes the decisions.
- `BMManager` applies one technique to the data in `bmI`; call `bmI.restore_BLD()` between experiments. Post-processing needs score-carrying prediction datasets.
- `BMGridSearch` runs baseline + all combinations for a model and returns the rows (pass a folder as `results_dir` to also save them as CSV).

**How to read the results**
- Post-processing (especially Equalized Odds and Reject Option) usually lowers 𝓕 the most, at some cost in balanced accuracy; Reweighing is mild and cheap in accuracy; combining pre + post rarely beats the best post-processor alone.
- Always compare against the `baseline` row of the *same model*; XGBoost and logistic regression start from different levels of bias.
- Caveats: results depend on the split seed and the sample; small validation groups make Equalized Odds unstable; a lower 𝓕 on one attribute does not guarantee fairness on another.

**Next:** [04_tradeoff_map.ipynb](04_tradeoff_map.ipynb) turns tables like the one above into a fairness-accuracy trade-off map.